# Pearl-Chat: Google Colab GPU / TPU Training

This notebook sets up the environment on Google Colab and runs pre-training of the Pearl-Chat native Luganda language model on GPU or TPU.

## 1. Environment installation on Colab

In [ ]:
!git clone https://github.com/wkambale/pearl-chat.git || true
%cd pearl-chat
!pip install --quiet flax>=0.10.0 optax orbax-checkpoint grain tokenizers datasets tqdm

## 2. Check accelerator (GPU or TPU)

In [ ]:
import jax
print("Default backend:", jax.default_backend())
print("Available devices:", jax.devices())

## 3. Hugging Face authentication (optional, for Sunbird/salt)

In [ ]:
import os
from huggingface_hub import login

# Provide your HF_TOKEN to access gated datasets like Sunbird/salt
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass

if hf_token:
    login(token=hf_token)
    print("Authenticated with Hugging Face")
else:
    print("Running without HF_TOKEN (open Wikipedia dataset will be used)")

## 4. Fetch data, train tokenizer, and prepare splits

In [ ]:
!python scripts/01_fetch_data.py
!python scripts/02_train_tokenizer.py --vocab-size 8192
!python scripts/03_prepare_dataset.py

## 5. Full pre-training on GPU / TPU

In [ ]:
!python scripts/04_train_pearlchat.py --mode full --steps 1000 --batch-size 32 --lr 5e-4

## 6. Test generation and chat demo

In [ ]:
!python scripts/05_run_chat_demo.py --prompt "Oli otya?" --max-tokens 64

## 7. Sharding visual demonstration

In [ ]:
!python scripts/06_sharding_visual_demo.py